In [ ]:
# Cell 1 — Install
!pip install -q "openenv-core[core]>=0.2.1"
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl transformers mergekit datasets accelerate peft bitsandbytes
!pip install -q torch-geometric
!git clone https://github.com/Godhand-Arnav/Scalar-finals.git /content/forge-rl

import sys, os, torch
sys.path.insert(0, '/content/forge-rl')
os.chdir('/content/forge-rl')

if not torch.cuda.is_available():
    raise RuntimeError('NO GPU DETECTED. Runtime -> Change runtime type -> T4 GPU')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Setup complete.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.6/174.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.6/728.6 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.3/152.3 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 15.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 25.0 MB

RuntimeError: NO GPU DETECTED. Runtime -> Change runtime type -> T4 GPU

In [ ]:
# Cell 2 — Load model
import torch

_is_p100 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] < 7

if _is_p100:
    print("Detected older GPU (Compute < 7.0). Using standard transformers + peft instead of Unsloth.")
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import get_peft_model, LoraConfig

    MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto"
    )
    model = get_peft_model(model, LoraConfig(
        r=16, lora_alpha=16,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
        lora_dropout=0, bias="none", task_type="CAUSAL_LM"
    ))
else:
    print("Detected Turing+ GPU. Using Unsloth for optimized training.")
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name='unsloth/Qwen2.5-3B-Instruct',
        max_seq_length=1024,
        load_in_4bit=True,
        dtype=None,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16, lora_alpha=16,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
        lora_dropout=0, bias='none',
        use_gradient_checkpointing='unsloth',
    )
print('Model loaded.')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.4.8 patched 36 layers with 36 QKV layers, 36 O layers and 0 MLP layers.


Model loaded.


In [ ]:
# Cell 3 — Reward function
import numpy as np
from env.misinfo_env import MisInfoForensicsEnv

# ACTIONS (canonical, from env/misinfo_env.py):
#   0=query_source   1=trace_origin    2=cross_reference  3=request_context
#   4=entity_link    5=temporal_audit  6=network_cluster  7=flag_manipulation
#   8=submit_verdict_real              9=submit_verdict_misinfo
#  10=submit_verdict_satire           11=submit_verdict_out_of_context
#  12=submit_verdict_fabricated
TOOL_ACTIONS = [0, 5, 2]  # query_source, temporal_audit, cross_reference

def _parse_verdict(text: str) -> int:
    t = text.lower()
    if any(w in t for w in ['fabricat']):
        return 12
    if any(w in t for w in ['out of context', 'recontextual', 'cropped']):
        return 11
    if any(w in t for w in ['satire', 'parody', 'joke', 'humor', 'comedic']):
        return 10
    if any(w in t for w in ['verified', 'legitimate', 'accurate', 'credible', 'true claim']):
        return 8
    if any(w in t for w in ['misinfo', 'false', 'manipulat', 'mislead', 'deceptive', 'disinformation']):
        return 9
    return 9  # default: misinfo

def _safe_step(env, action):
    result = env.step(action)
    if isinstance(result, tuple):
        if len(result) == 5:
            _, reward, terminated, truncated, _ = result
            return float(reward), bool(terminated or truncated)
        elif len(result) == 4:
            _, reward, terminated, _ = result
            return float(reward), bool(terminated)
        elif len(result) == 2:
            _, reward = result
            return float(reward), False
    return float(getattr(result, 'reward', 0.0)), bool(getattr(result, 'done', False))

def _safe_reset(env, seed=None):
    result = env.reset(seed=seed)
    if isinstance(result, tuple) and len(result) == 2:
        return result
    return result, {}

# ── Debug toggle — set False before final submission ─────────────────────────
REWARD_DEBUG = True
_debug_step = [0]  # mutable counter so it works inside the closure

def reward_fn(prompts, completions, claim_texts=None, **kwargs):
    if claim_texts is None:
        claim_texts = kwargs.get('claim_texts', None)
    rewards = []
    for i, completion in enumerate(completions):
        comp_text = (
            completion[-1]['content']
            if isinstance(completion, list)
            else str(completion)
        )
        try:
            env = MisInfoForensicsEnv(difficulty=1)
            obs, info = _safe_reset(env, seed=42 + i)

            done = False
            for act in TOOL_ACTIONS:
                if done:
                    break
                try:
                    _, done = _safe_step(env, act)
                except Exception:
                    break

            if not done:
                verdict_action = _parse_verdict(comp_text)
                try:
                    reward, _ = _safe_step(env, verdict_action)
                except Exception:
                    reward = 0.001
            else:
                verdict_action = -1  # episode closed early
                reward = 0.001

            clipped = float(np.clip(reward, 0.001, 0.999))
            rewards.append(clipped)

            # ── Debug line — logs first 30 steps then goes quiet ──────────────
            if REWARD_DEBUG and _debug_step[0] < 30:
                _debug_step[0] += 1
                comp_preview = comp_text[:70].replace('\n', ' ')
                default_flag = ' [DEFAULT]' if verdict_action == 9 and 'misinfo' not in comp_text.lower() else ''
                print(
                    f'[dbg #{_debug_step[0]:02d}] '
                    f'action={verdict_action}{default_flag} '
                    f'reward={clipped:.4f} | '
                    f'"{comp_preview}..."'
                )

        except Exception as e:
            print(f'[reward_fn] episode {i} error: {e}')
            rewards.append(0.001)

    return rewards

# ── Sanity check ──────────────────────────────────────────────────────────────
_LABEL_TO_ACTION = {
    'real': 8, 'verified': 8, 'true': 8,
    'misinfo': 9, 'misinformation': 9, 'false': 9,
    'satire': 10, 'parody': 10,
    'out_of_context': 11, 'out of context': 11,
    'fabricated': 12,
}

print('reward_fn ready. Running sanity check (5 seeds)...')
_passed, _results = 0, []

for _seed in range(5):
    try:
        _env = MisInfoForensicsEnv(difficulty=1)
        _, _info = _safe_reset(_env, seed=_seed)
        _label = (
            getattr(getattr(_env, 'graph', None), 'true_label', None) or
            _info.get('true_label') or 'misinfo'
        )
        _action = _LABEL_TO_ACTION.get(str(_label).lower().strip(), 9)
        _done = False
        for _act in TOOL_ACTIONS:
            if _done: break
            try: _, _done = _safe_step(_env, _act)
            except: break
        if not _done:
            _r, _ = _safe_step(_env, _action)
            _results.append((_seed, str(_label), _action, _r))
            if _r > 0.5: _passed += 1
        else:
            _results.append((_seed, str(_label), _action, 0.0))
    except Exception as _e:
        _results.append((_seed, 'error', -1, 0.0))
        print(f'  seed {_seed} error: {_e}')

for _seed, _label, _action, _r in _results:
    print(f'  {"✅" if _r > 0.5 else "❌"} seed={_seed} label={_label} action={_action} reward={_r:.4f}')

if _passed >= 3:
    print(f'\n✅ PASSED ({_passed}/5) — reward_fn working. Safe to train.')
elif _passed >= 1:
    print(f'\n⚠️  PARTIAL ({_passed}/5) — reward_fn reaches env. Check env task balance.')
else:
    print(f'\n❌ FAILED (0/5) — DO NOT train.')

reward_fn ready. Running sanity check (5 seeds)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✅ seed=0 label=real action=8 reward=0.6400
  ❌ seed=1 label=out_of_context action=11 reward=0.3550
  ✅ seed=2 label=real action=8 reward=0.6400
  ✅ seed=3 label=fabricated action=12 reward=0.5134
  ✅ seed=4 label=fabricated action=12 reward=0.5167

✅ PASSED (4/5) — reward_fn working. Safe to train.


In [ ]:
# Cell 4 — Dataset
from datasets import Dataset
from env.forge_env import ForgeEnv, ForgeEnvConfig  # ← add this line
import random
from datasets import Dataset
import random

PROMPT_TMPL = ('You are a misinformation forensics agent. Investigate the claim:\n'
               'CLAIM: {claim}\n\n'
               'Submit your verdict:\n'
               '- "This is MISINFO because [reason]"\n'
               '- "This is SATIRE because [reason]"\n'
               '- "This is VERIFIED because [reason]"\n'
               '- "This is FABRICATED because [reason]"\n'
               '- "This is OUT OF CONTEXT because [reason]"\n'
               'Your verdict:')

TASK_NAMES = ['fabricated_stats', 'out_of_context', 'coordinated_campaign',
              'satire_news', 'verified_fact', 'politifact_liar',
              'image_forensics', 'sec_fraud']

def _get_claim(task, seed):
    try:
        cfg = ForgeEnvConfig(budget=3, seed=seed)
        env = ForgeEnv(cfg)
        _, info = _safe_reset(env)
        for attr in ('_claim_text', 'claim_text'):
            val = getattr(env, attr, None)
            if val: return str(val)
        if info and 'claim_text' in info: return info['claim_text']
    except: pass
    return f'Unverified claim #{seed} in domain: {task}'

random.seed(42)
claims, prompts = [], []
for i in range(200):
    claim = _get_claim(TASK_NAMES[i % len(TASK_NAMES)], seed=i)
    claims.append(claim)
    prompts.append([{'role': 'user', 'content': PROMPT_TMPL.format(claim=claim)}])

dataset = Dataset.from_dict({'prompt': prompts, 'claim_texts': claims})
print(f'Dataset: {len(dataset)} rows')

Dataset: 200 rows


In [ ]:
# Cell 5 — Baseline evaluation
import numpy as np
from env.misinfo_env import MisInfoForensicsEnv

_LABEL_TO_VERDICT = {
    'real': 8, 'verified': 8,
    'misinfo': 9, 'misinformation': 9, 'false': 9,
    'satire': 10, 'parody': 10,
    'out_of_context': 11,
    'fabricated': 12,
}

def evaluate_heuristic(n_episodes=20, difficulty=1):
    rewards, correct = [], 0
    print(f'Evaluating {n_episodes} episodes...')

    for i in range(n_episodes):
        try:
            env = MisInfoForensicsEnv(difficulty=difficulty)
            obs, info = _safe_reset(env, seed=100 + i)

            # Get true label → pick correct verdict
            true_label = (
                getattr(getattr(env, 'graph', None), 'true_label', None)
                or info.get('true_label')
                or 'misinfo'
            )
            verdict_action = _LABEL_TO_VERDICT.get(
                str(true_label).lower().strip(), 9
            )

            # Run investigation tools
            done = False
            for act in TOOL_ACTIONS:
                if done: break
                try:
                    _, done = _safe_step(env, act)
                except Exception:
                    break

            # Submit verdict
            reward = 0.001
            if not done:
                try:
                    reward, _ = _safe_step(env, verdict_action)
                except Exception:
                    reward = 0.001

            clipped = float(np.clip(reward, 0.001, 0.999))
            rewards.append(clipped)
            if clipped > 0.5:
                correct += 1

            if (i + 1) % 5 == 0:
                print(f'  ep {i+1}/{n_episodes} | '
                      f'label={true_label} action={verdict_action} '
                      f'reward={clipped:.4f} | '
                      f'running_mean={np.mean(rewards):.4f}')

        except Exception as e:
            print(f'  ep {i} error: {e}')
            rewards.append(0.001)

    mean_r = float(np.mean(rewards))
    acc    = correct / n_episodes
    print(f'\nHEURISTIC BASELINE — reward: {mean_r:.4f} | acc: {acc:.2%}')
    return mean_r, acc

baseline_reward, baseline_acc = evaluate_heuristic(n_episodes=20, difficulty=1)

Evaluating 20 episodes...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ep 20/20 | label=misinfo action=9 reward=0.6067 | running_mean=0.6131

HEURISTIC BASELINE — reward: 0.6131 | acc: 100.00%


In [ ]:
# Cell 6 — GRPO Training
import torch as _th, inspect
from trl import GRPOConfig, GRPOTrainer

_gpu = _th.cuda.get_device_name(0) if _th.cuda.is_available() else ''
_use_bf16 = any(x in _gpu for x in ['A100', 'A10', 'H100', 'RTX 30', 'RTX 40'])
_use_fp16 = not _use_bf16 and _th.cuda.is_available()
print(f'GPU: {_gpu} | bf16={_use_bf16} | fp16={_use_fp16}')

_valid = set(inspect.signature(GRPOConfig).parameters.keys())

_cfg = dict(
    output_dir='./forge-grpo',
    max_steps=200,                      # FIX: 150→200, more learning signal
    num_generations=4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,      # effective batch = 8
    learning_rate=5e-6,
    num_generations=4,
    generation_batch_size=4,
    save_steps=50,
    logging_steps=5,
    report_to='none',
    warmup_ratio=0.1,
    bf16=_use_bf16,
    fp16=_use_fp16,
)

# max_completion_length (TRL 1.x) vs max_new_tokens (TRL 0.x) — set whichever exists
for k, v in [('max_completion_length', 128), ('max_new_tokens', 128)]:
    if k in _valid:
        _cfg[k] = v
        break

# temperature — set if supported, else model.generate() uses default 1.0
if 'temperature' in _valid:
    _cfg['temperature'] = 0.7

_cfg['num_generations'] = 4
_cfg['generation_batch_size'] = 4
config = GRPOConfig(**_cfg)
print(f'GRPOConfig created ({len(_cfg)} params)')
print(f'  max_steps={config.max_steps} | '
      f'num_generations={config.num_generations} | '
      f'effective_batch={config.per_device_train_batch_size * config.gradient_accumulation_steps}')

# Dual init — handles processing_class (TRL 1.x) vs tokenizer (TRL 0.x)
try:
    trainer = GRPOTrainer(
        model=model,
        reward_funcs=reward_fn,
        args=config,
        train_dataset=dataset,
        processing_class=tokenizer,     # TRL 1.x
    )
    print('GRPOTrainer init: processing_class path')
except TypeError:
    trainer = GRPOTrainer(
        model=model,
        reward_funcs=reward_fn,
        args=config,
        train_dataset=dataset,
        tokenizer=tokenizer,            # TRL 0.x fallback
    )
    print('GRPOTrainer init: tokenizer path')

print(f'\nStarting GRPO training — {config.max_steps} steps (~60 min on T4)...')
print('Watch [dbg] lines from reward_fn for live reward signal.\n')
trainer.train()
print('\nTraining complete.')

ModuleNotFoundError: No module named 'trl'

In [ ]:
# Cell 7 — Evaluation + plots
import matplotlib.pyplot as plt
import matplotlib; matplotlib.use('Agg')
import os
import numpy as np
from env.misinfo_env import MisInfoForensicsEnv
os.makedirs('results', exist_ok=True)

# FIX: use MisInfoForensicsEnv not ForgeEnv — no network, no hang
def evaluate_trained(n_episodes=20, difficulty=1):
    rewards, correct = [], 0
    if not _is_p100:
        FastLanguageModel.for_inference(model)  # switch to inference mode once
    print(f'Evaluating trained model ({n_episodes} episodes)...')

    for i in range(n_episodes):
        try:
            # FIX: get claim from local env, not ForgeEnv
            env = MisInfoForensicsEnv(difficulty=difficulty)
            obs, info = _safe_reset(env, seed=200 + i)

            claim = (
                getattr(env, '_claim_text', None)
                or getattr(env, 'claim_text', None)
                or info.get('claim_text', f'Claim #{200+i}')
            )

            # Generate verdict from trained LLM
            prompt_ids = tokenizer.apply_chat_template(
                [{'role': 'user', 'content': PROMPT_TMPL.format(claim=claim)}],
                tokenize=True, return_tensors='pt', add_generation_prompt=True,
            ).to('cuda')
            outputs = model.generate(
                input_ids=prompt_ids, max_new_tokens=80,
                temperature=0.3, do_sample=True
            )
            response = tokenizer.decode(
                outputs[0][prompt_ids.shape[1]:], skip_special_tokens=True
            )

            # Run tools then submit LLM verdict — same env, no new ForgeEnv
            done = False
            for act in TOOL_ACTIONS:
                if done: break
                try:
                    _, done = _safe_step(env, act)
                except Exception:
                    break

            reward = 0.001
            if not done:
                try:
                    reward, _ = _safe_step(env, _parse_verdict(response))
                except Exception:
                    reward = 0.001

            clipped = float(np.clip(reward, 0.001, 0.999))
            rewards.append(clipped)
            if clipped > 0.5: correct += 1

            if (i + 1) % 5 == 0:
                print(f'  ep {i+1}/{n_episodes} | '
                      f'verdict={_parse_verdict(response)} '
                      f'reward={clipped:.4f} | '
                      f'running_mean={np.mean(rewards):.4f}')

        except Exception as e:
            print(f'  trained eval error ep {i}: {e}')
            rewards.append(0.001)

    return float(np.mean(rewards)), correct / n_episodes

trained_reward, trained_acc = evaluate_trained()
print(f'\nBASELINE — {baseline_reward:.4f} | {baseline_acc:.2%}')
print(f'TRAINED  — {trained_reward:.4f} | {trained_acc:.2%}')
print(f'DELTA    — {trained_reward - baseline_reward:+.4f} | {trained_acc - baseline_acc:+.2%}')

# ── Extract reward curve from trainer logs ────────────────────────────────────
steps, rew = [], []
REWARD_LOG_KEYS = (
    'rewards/mean', 'reward/mean', 'reward',
    'train/reward', 'rewards/reward_mean', 'mean_reward',
)
for l in trainer.state.log_history:
    if 'step' not in l: continue
    for key in REWARD_LOG_KEYS:
        if key in l:
            steps.append(l['step'])
            rew.append(l[key])
            break

# ── Plot 1: reward curve ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
if steps:
    ax.plot(steps, rew, 'b-', linewidth=2.5, label='Training reward')
else:
    print('WARNING: no reward keys found in log_history — curve empty')
ax.axhline(y=baseline_reward, color='r', linestyle='--', linewidth=2,
           label=f'Heuristic baseline ({baseline_reward:.3f})')
ax.set_xlabel('Training Step'); ax.set_ylabel('Mean Episode Reward')
ax.set_title('FORGE-MA: LLM Agent Learning Misinformation Detection via GRPO',
             fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig('results/reward_curve.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: results/reward_curve.png')

# ── Plot 2: before/after bar — FIX: was missing, README references it ─────────
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(
    ['Heuristic Baseline', 'GRPO Trained'],
    [baseline_reward, trained_reward],
    color=['#e74c3c', '#2ecc71'],
    edgecolor='black', linewidth=1.2, width=0.5
)
ax.set_ylabel('Mean Episode Reward', fontsize=13)
ax.set_title('Before vs After GRPO Training\n(FORGE-MA Misinformation Agent)', fontsize=13)
ax.set_ylim(0, 1.0)
for bar, val in zip(bars, [baseline_reward, trained_reward]):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/before_after.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: results/before_after.png')
print('\nCOMMIT BOTH PNG FILES BEFORE SUBMITTING')